In [1]:
import zarr
import time
import numpy as np
from concurrent.futures import ThreadPoolExecutor

def benchmark_zarr_read(dataset_path, num_workers=8):
    root = zarr.open(dataset_path, mode="r")
    data = root["data"]
    
    # Identify chunks or splits for workers
    # Assuming we split by the first dimension
    total_size = data.shape[0]
    chunk_size = total_size // num_workers
    ranges = [(i * chunk_size, (i + 1) * chunk_size) for i in range(num_workers)]
    # Handle remainder
    ranges[-1] = (ranges[-1][0], total_size)

    def read_chunk(start_end):
        start, end = start_end
        # Crucial: Perform the actual read into memory
        _ = data[start:end] 
        return True

    print(f"Starting read with {num_workers} workers...")
    
    start_time = time.perf_counter()
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        list(executor.map(read_chunk, ranges))
    end_time = time.perf_counter()

    duration = end_time - start_time
    print(f"Total time: {duration:.4f} seconds")
    return duration


In [2]:
benchmark_zarr_read("/mnt/tier1/project/p200177/DE_371_bis/meps-2p5km-2020-2025-1h-v2_subdomain_subvars_rechu.zarr", num_workers=8)

Starting read with 8 workers...
Total time: 268.9392 seconds


268.9391676579835